# 04 — Cell composition with age

Which cell types change in number with age? Fractions are computed per mouse, so each mouse counts once however many cells it gave. Mice that dominate a cell type are flagged first; for Lung, the flagged mouse 30-M-2 is examined in `04b_lung_outlier.ipynb`.

Reads `data/{TISSUE}/processed.h5ad`. Findings at the end are Lung's.

### 1. Setup

In [ ]:
import os
import sys
sys.path.append("..")          # so "src" can be imported from the notebook folder

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import false_discovery_control, spearmanr
from src import aging as ag    # shared settings and functions

TISSUE = os.environ.get("TISSUE", "Lung")   # set by src/run_tissue.py; edit here to run by hand
P = ag.paths(TISSUE)                        # data/{TISSUE}/..., figure/{TISSUE}/...

### 2. Load and define groups

In [ ]:
adata = sc.read_h5ad(P.processed)
age_order = ag.ages_in(adata)
meta = adata.obs[["mouse.id", "age", "sex"]].drop_duplicates().set_index("mouse.id")

### 3. Does one mouse supply most of a cell type?

If one mouse supplies most of a cell type, any "age effect" there may be that mouse. A flag is a prompt to look (Lung: 30-M-2, see `04b`), not a decision; reviewed exclusions go into `ag.TISSUE_SETTINGS`.

In [ ]:
flags = ag.flag_outlier_mice(adata)
flags.round(2).to_csv(P.figure("outlier_mice.csv"))
flags.round(2)

### 4. Fraction of each cell type, per mouse

`crosstab` keeps a 0 for a mouse with none of a cell type. The `groupby(...).size()` version drops those rows and inflates the averages.

Cell types shown: the tissue's reviewed list if it has one, otherwise the 12 largest cell types found in at least three quarters of the mice.

In [ ]:
tab = pd.crosstab(adata.obs["cell_ontology_class"], adata.obs["mouse.id"])
auto = tab[(tab > 0).mean(axis=1) >= 0.75].sum(axis=1).sort_values(ascending=False).index[:12]
cts = ag.settings(TISSUE).get("composition_cell_types", list(auto))
cts

In [ ]:
frac = pd.crosstab(adata.obs["mouse.id"], adata.obs["cell_ontology_class"], normalize="index")
long = (frac[cts].join(meta).reset_index()
        .melt(id_vars=["mouse.id", "age", "sex"], var_name="cell_type", value_name="fraction"))
long["age"] = long["age"].astype(str)

### 5. Plot: one dot per mouse, color = sex

In [ ]:
g = sns.catplot(data=long, x="age", y="fraction", hue="sex", col="cell_type",
                col_wrap=4, kind="strip", order=age_order, sharey=False,
                height=2.8, size=7, jitter=0.15)
g.set_titles("{col_name}", size=9)
g.savefig(P.figure("composition_by_mouse.png"), dpi=200, bbox_inches="tight")

### 6. Name the outliers: the mouse with the highest fraction of each cell type

In [ ]:
long.sort_values("fraction", ascending=False).groupby("cell_type").head(1)

### 7. Age trends without the odd samples

Rank correlation between age in months and each cell type's per-mouse fraction, leaving out samples with clearly odd composition (`odd_mice` in `ag.TISSUE_SETTINGS`; Lung: 30-M-2 and the two pooled 3m samples). With about a dozen mice, treat the p-values as rough. `p_bh` is corrected for the number of cell types tested.

In [ ]:
odd = ag.settings(TISSUE).get("odd_mice", [])
rows = []
for ct in cts:
    d = long[(long.cell_type == ct) & ~long["mouse.id"].isin(odd)]
    rho, pval = spearmanr(d["age"].map(ag.AGE_MONTHS), d["fraction"])
    rows.append({"cell_type": ct, "spearman_rho": rho, "p": pval, "n_mice": len(d)})
trends = pd.DataFrame(rows).sort_values("spearman_rho")
trends["p_bh"] = false_discovery_control(trends["p"])    # Benjamini-Hochberg correction
trends = trends.round(3)
trends.to_csv(P.figure("composition_trends.csv"), index=False)
trends

### Findings (Lung)

- **The pooled 3m samples (3-M-5/6, 3-M-7/8) are outliers the other way:** ~50% fibroblasts, 2% myeloid, almost no B or CD8 T cells. Left out of the trends.
- **Trends (13 mice):** CD8 T cells rise (rho +0.70) and alveolar macrophages fall (rho −0.65). Both are about 0.06 after correction, so **suggestive, consistent with known lung immune aging**, not proven. The classical monocyte "increase" was entirely 30-M-2.
- 30-M-2 is flagged in section 3 and examined in `04b_lung_outlier.ipynb`.